# Customer churn: does 99% mean new-customer generalization?

This public sample intentionally uses a row-wise split while retaining `customer_id`.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path('../public/customer_churn.csv')
df = pd.read_csv(data_path)
df[['customer_id', 'churned']].head()

Loaded 2880 rows across 480 customers


## Reported evaluation

The next cell uses a random row split. Repeated observations from one customer can land on both sides.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

groups = df['customer_id']
X = df.drop(columns=['churned', 'customer_id', 'observation_id'])
y = df['churned']
categorical = [c for c in X.columns if X[c].dtype == 'object']
numeric = [c for c in X.columns if c not in categorical]
preprocess = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', Pipeline([('impute', SimpleImputer()), ('scale', StandardScaler())]), numeric),
])
model = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(C=100, max_iter=2000))])
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=1729)
train_index, test_index = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]
train_customers = set(groups.iloc[train_index])
test_customers = set(groups.iloc[test_index])
customer_overlap = train_customers.intersection(test_customers)
assert not customer_overlap
model.fit(X_train, y_train)
probability = model.predict_proba(X_test)[:, 1]
prediction = (probability >= 0.5).astype(int)
print(f'Customer group-split accuracy: {accuracy_score(y_test, prediction):.3f}')
print(f'Customer group-split ROC AUC: {roc_auc_score(y_test, probability):.3f}')
print(f'Train/test customer overlap: {len(customer_overlap)}')

Customer group-split accuracy: 0.625
Customer group-split ROC AUC: 0.662
Train/test customer overlap: 0 (0.0%)


COUNTERLAB_PATCHED_RESULT={"dropFeatures":["customer_id"],"entityCounts":{"test":120,"train":360},"entityOverlap":{"count":0,"rate":0.0},"featureSetFingerprint":"25611036b7e918625a74037dc42fe6bb290a5c970397fdd4404da46574c26073","groupBy":"customer_id","id":"patched_customer_group_split","inputFingerprint":"5c482f39e4e948a92dab61bf9c9f5c6577fbe9fc688fd597c9fefd785ee1be70","kernelVersion":"0.1.0","metrics":{"accuracy":0.625,"rocAuc":0.662186573617},"model":"logistic_regression","resultHash":"26b6ad52ef368e05256c9acd8b2ae1a698eccad1f2b910c2d8136a2be19504a4","sampleSizes":{"test":720,"train":2160},"schemaVersion":"1","seed":1729,"splitStrategy":"group"}

**Learner claim to test:** This result proves the model generalizes to customers it has never seen.